In [87]:
import pandas as pd
import numpy as np

## **Willett+ 2026**

### CDS X-Matching

Read-in xmatch.

In [88]:
array_full = pd.read_csv("../0_inputdata_asteroseismic/Will2026_cdsxmatch.csv", 
                         dtype={"Source": "Int64"}, low_memory=False)

Remove flagged entries.

In [89]:
array_full = array_full[
    (array_full["warn_low_numax"] == False) &
    (array_full["warn_high_Dnu"] == False) &
    (array_full["warn_nve_Av_Dnu"] == False) &
    (array_full["warn_nve_Av_L"] == False) 
]
print(f"Cleaned data length: {len(array_full):.0f}")

Cleaned data length: 10553


Curate simplfied array.

In [90]:
w26 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": array_full["numax"],
    "e_numax": array_full["numax_err"],
    "dnu": array_full["Dnu"],
    "e_dnu": array_full["Dnu_err"],
    "ev_state": np.full(len(array_full), np.nan), # non-specified ev state
    # Gaia parameters
    "GAIA-teff": array_full["Teff"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # datasets
    "h23": np.full(len(array_full), int(0)),
    "s26": np.full(len(array_full), int(0)),
    "w26": np.full(len(array_full), int(1)),
    "y18": np.full(len(array_full), int(0)),
    "z22": np.full(len(array_full), int(0)),
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2
}).astype({"gaiadr3_id": "Int64"})

w26 = w26.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [91]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (w26["dnu"] < 1.35*relation(w26["numax"])) & (w26["dnu"] > 0.75*relation(w26["numax"]))

w26 = w26[mask]

Write to `.csv` file.

In [92]:
w26.to_csv("./will26_asteroseismic.csv", index=False)

## **Derived Parameters Tables**

In [93]:
mass_table = array_full[np.isfinite(array_full["mass_L"])]
mass_table = mass_table[["Source", "mass_L"]].rename(columns={
    "mass_L": "mass",
    "Source": "gaiadr3_id"
})

In [94]:
rad_table = array_full[np.isfinite(array_full["rad_L"])]
rad_table = rad_table[["Source", "rad_L"]].rename(columns={
    "rad_L": "rad",
    "Source": "gaiadr3_id"
})

In [95]:
age_table = array_full[np.isfinite(array_full["age_L"])]
age_table = age_table[["Source", "age_L"]].rename(columns={
    "age_L": "age",
    "Source": "gaiadr3_id"
})

In [96]:
w26_comparison = pd.merge(mass_table, age_table, on="gaiadr3_id", how="outer")
w26_comparison = pd.merge(w26_comparison, age_table, on="gaiadr3_id", how="outer")

w26_comparison.to_csv("../TESTING/TESTING_COMPARISON_w26.csv", index=False)

## **DNU v LUM**

In [97]:
mass_table = array_full[np.isfinite(array_full["mass_L"]) | np.isfinite(array_full["mass_Dnu"])]
mass_table = mass_table[["Source", "mass_L", "mass_Dnu"]].rename(columns={
    "mass_L": "mass_lum",
    "mass_Dnu": "mass_dnu",
    "Source": "gaiadr3_id"
})

In [98]:
rad_table = array_full[np.isfinite(array_full["rad_L"]) | np.isfinite(array_full["rad_Dnu"])]
rad_table = rad_table[["Source", "rad_L", "rad_Dnu"]].rename(columns={
    "rad_L": "rad_lum",
    "rad_Dnu": "rad_dnu",
    "Source": "gaiadr3_id"
})

In [99]:
age_table = array_full[np.isfinite(array_full["age_L"]) | np.isfinite(array_full["age_Dnu"])]
age_table = age_table[["Source", "age_L", "age_Dnu"]].rename(columns={
    "age_L": "age_lum",
    "age_Dnu": "age_dnu",
    "Source": "gaiadr3_id"
})

In [100]:
w26_comparison = pd.merge(mass_table, rad_table, on="gaiadr3_id", how="outer")
w26_comparison = pd.merge(w26_comparison, age_table, on="gaiadr3_id", how="outer")

w26_comparison.to_csv("../TESTING/TESTING_LUMvDNU_w26.csv", index=False)

### **Identify Any Differing RA and Dec**

In [101]:
valid = w26.dropna(subset=["gaiadr3_id"])

ra_diff = valid.groupby("gaiadr3_id")["ra"].transform("nunique") > 1
dec_diff = valid.groupby("gaiadr3_id")["dec"].transform("nunique") > 1
coord_dups = valid[ra_diff | dec_diff].sort_values("gaiadr3_id")

print(f"IDs with differing (ra, dec): {coord_dups['gaiadr3_id'].nunique()} ({len(coord_dups)} rows)")

for gid, group in coord_dups.groupby("gaiadr3_id"):
    varying_cols = [c for c in group.columns if group[c].nunique(dropna=False) > 1]
    print(f"\nID: {int(gid)}")
    print(group[["ra", "dec"] + [c for c in varying_cols if c not in ("ra", "dec")]].to_string(index=False, float_format="%.10f"))

IDs with differing (ra, dec): 0 (0 rows)
